In [ ]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine

# 1. Load the dataset
df = pd.read_csv('US_Top_50_Universities_2026.csv')

# 2. Standardize Column Names (Lowercase & Underscores)
df.columns = [col.strip().lower().replace(' ', '_') for col in df.columns]

print("--- 🔍 STARTING DATA VALIDATION ---")

# 3. Clean Trailing/Leading Spaces in text data
# This prevents "Private " being treated differently than "Private"
string_cols = df.select_dtypes(include=['object']).columns
for col in string_cols:
    df[col] = df[col].str.strip()
print(f"✅ Cleaned whitespace from columns: {list(string_cols)}")

# 4. Check for Missing Values (NaN)
missing_count = df.isnull().sum().sum()
if missing_count == 0:
    print("✅ No missing values (NaN) found.")
else:
    print(f"⚠️ Warning: Found {missing_count} missing values.")
    # df = df.dropna() # Uncomment this if you want to remove them

# 5. Check for 0.0 values where they shouldn't be
# (Rank, Year, and Scores should never be 0)
cols_to_check_zero = ['national_rank', 'founded_year', 'research_impact_score', 'employment_rate']
for col in cols_to_check_zero:
    zero_count = (df[col] == 0).sum()
    if zero_count > 0:
        print(f"⚠️ Warning: Column '{col}' contains {zero_count} zero values.")
    else:
        print(f"✅ Column '{col}' has no zero values.")

# 6. Enforce Data Types
try:
    df['national_rank'] = df['national_rank'].astype(int)
    df['founded_year'] = df['founded_year'].astype(int)
    
    float_cols = ['research_impact_score', 'intl_student_ratio', 'employment_rate']
    for col in float_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype(float)
        
    print("✅ All data types enforced (Ints and Floats).")
except Exception as e:
    print(f"❌ Error during type conversion: {e}")

print("--- 🚀 VALIDATION COMPLETE ---\n")

# 7. Final Step: Upload to SQL
engine = create_engine('sqlite:///uni_data.db')
df.to_sql('universities', con=engine, if_exists='replace', index=False)
print("📦 Data successfully pushed to 'uni_data.db'")

"""
📊 Quick Tip for your "Report"
Run this one line in Python to get all the stats you need for your Word document:

Python
print(df.describe())
"""

--- 🔍 STARTING DATA VALIDATION ---
✅ Cleaned whitespace from columns: ['university_name', 'institution_type', 'state']
✅ No missing values (NaN) found.
✅ Column 'national_rank' has no zero values.
✅ Column 'founded_year' has no zero values.
✅ Column 'research_impact_score' has no zero values.
✅ Column 'employment_rate' has no zero values.
✅ All data types enforced (Ints and Floats).
--- 🚀 VALIDATION COMPLETE ---

📦 Data successfully pushed to 'uni_data.db'
